In [1]:
!pip install -q transformers datasets accelerate sentencepiece

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from torch.utils.data import Dataset

In [3]:
df = pd.read_csv("/kaggle/input/datasets/satishgunjal/grammar-correction/Grammar Correction.csv")

In [4]:
df.head()

,Serial Number,Error Type,Ungrammatical Statement,Standard English
0,1,Verb Tense Errors,I goes to the store everyday.,I go to the store everyday.
1,2,Verb Tense Errors,They was playing soccer last night.,They were playing soccer last night.
2,3,Verb Tense Errors,She have completed her homework.,She has completed her homework.
3,4,Verb Tense Errors,He don't know the answer.,He doesn't know the answer.
4,5,Verb Tense Errors,The sun rise in the east.,The sun rises in the east.


In [5]:
train_df, val_df = train_test_split(df, test_size = 0.2, random_state = 42)

In [6]:
#Tokeization
tokenizer = AutoTokenizer.from_pretrained("t5-small")

In [7]:
class GECDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len = 128):
        self.data = dataframe.reset_index(drop = True)
        self.tokenizer = tokenizer
        self.max_len= max_len
    def __len__(self):
        return len(self.data)
    def __getitem__(self, index):
        row = self.data.iloc[index]
        input_text = "gec: " + str(row["Ungrammatical Statement"])
        target_text = str(row["Standard English"])
        inputs = self.tokenizer(input_text, max_length = self.max_len, padding = 'max_length', truncation = True, return_tensors = "pt")
        targets = self.tokenizer(target_text, max_length = self.max_len, padding = 'max_length', truncation = True, return_tensors = "pt")
        labels = targets["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": labels
        }

In [8]:
train_dataset = GECDataset(train_df, tokenizer)
val_dataset = GECDataset(val_df, tokenizer)
print("Data Loading Done! Train Data Size:", len(train_dataset))

Data Loading Done! Train Data Size: 1614


In [9]:
#Working with our model
from transformers import T5ForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments

model = T5ForConditionalGeneration.from_pretrained("t5-small")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [10]:
import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [11]:
#Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir = "./gec_t5_results",
    eval_strategy = "epoch",
    learning_rate = 3e-4,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    num_train_epochs = 3,
    predict_with_generate = True,
    fp16 = True,
    report_to = "none"
)

trainer = Seq2SeqTrainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [12]:
#training
trainer.train()

model.save_pretrained("./saved_gec_model4")
tokenizer.save_pretrained("./saved_gec_model4")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,No log,0.563928
2,No log,0.507520
3,No log,0.506194


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_gec_model4/tokenizer_config.json',
 './saved_gec_model4/tokenizer.json')

In [13]:
#test the core logic
test_tokenizer = AutoTokenizer.from_pretrained("/kaggle/working/saved_gec_model4")
test_model = T5ForConditionalGeneration.from_pretrained("/kaggle/working/saved_gec_model4")

def correct_my_grammar(wrong_text):
    input_text = "gec: " + wrong_text
    inputs = test_tokenizer(input_text, padding = "longest", return_tensors = "pt")
    outputs = test_model.generate(**inputs, max_length = 128)
    corrected_text = test_tokenizer.decode(outputs, skip_special_tokens = True)
    return corrected_text

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [20]:
test_case ="Although he were tired but he still decidedto walking all the way to home"
print(f"Right sentence: {correct_my_grammar(test_case)}")

Right sentence: ['Although he was tired, he still decided to walk all the way to home.']
